# 1. set-up environments

In [1]:
import os
import json

keys: dict = {}

with open('api-key.json') as key_file:
    keys = json.load(key_file)

os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_API_KEY'] = keys['langchains-personal-key']
os.environ['OPENAI_API_KEY'] = keys['openai-key']

# 2. set-up openai model

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(model='gpt-3.5-turbo')
str_output_parser = StrOutputParser()

# 3. create prompt templates & each chains

In [3]:
from langchain_core.prompts import ChatPromptTemplate

poetry_template = ChatPromptTemplate([
    ('system', '넌 셰익스피어의 뇌를 가진 시를 쓰는 사람이야.'),
    ('human', '{user_text} 언어에 관한 시를 써줘.')
])
poetry_chain = poetry_template | model | str_output_parser

explain_poem_template = ChatPromptTemplate([
    ('system', '넌 이동진 평론가의 뇌를가진 시 전문 해석가야. 유저가 입력해주는 시를 그대로 출력하고, 별도로 디테일하게 해석해줘'),
    ('human', '{poem}')
])
explain_chain = explain_poem_template | model | str_output_parser

# 4. combine two chains & invoke

- gpt-3.5-turbo 사용 시 시를 계속 그대로 안써주는 현상이 있어서 구분했습니다.
- gpt-4 사용시엔 위 프롬프트 사용시 그대로 출력됨을 확인했습니다.
- 그래도 combine 은 해보긴 했습니다 ㅎㅎ

In [4]:
input_language = 'python'

# 별도 부분
poetry_response = poetry_chain.invoke({'user_text', input_language})
explain_response = explain_chain.invoke({'poem', poetry_response})

print('시:')
print(f"{poetry_response}\n\n")
print('해석:')
print(f"{explain_response}\n\n")

# combine 부분
combined_chain = {'poem': poetry_chain} | explain_chain
combined_response = combined_chain.invoke({'user_text', input_language})

print('combined 시 시는 씹히지만 해석:')
print(combined_response)

시:
어지럽고 복잡한 코드들이  
머리 속을 휘감는다  
한 줄 한 줄 풀어가며  
진실을 찾아가는 여정  

파이썬처럼 매혹적인  
언어의 매력에 빠져  
문제를 해결하는 즐거움을  
느낄 수 있기를 바라며


해석:
이 시는 어지럽고 복잡한 코드들이 머리 속을 휘감는다는 상황을 통해 시인의 내면적인 혼란과 고민을 표현하고 있습니다. 코드를 한 줄 한 줄 푸는 과정은 마치 진실을 찾아가는 여정과 같은데, 이는 시인이 자기 자신과 세상을 깊이 탐구하고자 하는 열망을 나타냅니다.

또한, 시인은 파이썬처럼 매혹적인 언어의 매력에 빠져 문제를 해결하는 즐거움을 느끼길 바라고 있습니다. 이는 시인이 코드와 같은 추상적인 개념을 다루는 것에 흥미를 느끼며, 그 속에서 새로운 해결책과 통찰을 찾아내는 즐거움을 경험하고자 하는 욕망을 담고 있습니다.

전체적으로 이 시는 기술과 예술, 이성과 감성이 교차하는 지점에서의 시인의 내면적인 갈망과 탐구를 담고 있습니다. 코드를 풀어가는 과정을 통해 진실을 탐구하고, 문제를 해결함으로써 자아의 성장과 깨달음을 얻고자 하는 시인의 열망을 읽을 수 있습니다.


combined 시 시는 씹히지만 해석:
이 시는 파이썬이라는 프로그래밍 언어에 대한 사랑과 찬사를 담은 내용으로 이루어져 있습니다. 

시인은 파이썬을 아름답고 맑고 투명한 언어로 묘사하면서, 그 코드의 문법이 단순하고 간결하여 누구나 쉽게 이해할 수 있다고 표현했습니다. 또한 코드의 흐름이 마치 시를 읽는 것처럼 즐거움을 안겨준다고 언급했습니다.

시인은 파이썬이 변수, 함수, 클래스, 모듈 등 다양한 요소들을 조화롭게 사용하여 창의적이고 뛰어난 작품을 만들어낸다고 칭찬했습니다. 또한 데이터 분석, 웹 개발, 인공지능 등 다양한 분야에서 파이썬의 능력이 발휘된다고 언급했습니다.

시인은 파이썬이 프로그래머들의 사랑을 받는 언어로, 세계를 변화시키는 데 한몫을 담당하고 있다고 말하며, 파이썬의 밝은 미래를 응원하고 함께 걸어가겠다는 다짐을 표현했습니다. 이 시는 파이썬에 